# 1 · Data ingestion, ETL & SQL | 45 minutes
**Business mission:** clean a small sample of real Kaggle customer-support tickets and load them into your **team-owned Supabase tables**. Then answer business questions with SQL.

Source: [Customer Support Ticket Dataset (Kaggle)](https://www.kaggle.com/datasets/suraj520/customer-support-ticket-dataset). **Instructor already downloaded + ingested it**; no Kaggle account or local CSV required.

> A source caveat: `First Response Time` and `Time to Resolution` are timestamps. Their difference is **time from first response to resolution**, not total ticket lifetime or a contractual SLA. Missing durations stay null. Dataset is synthetic and may have noisy text.

In [1]:
!pip install -q "psycopg[binary]>=3.2,<4" "pandas>=2.2,<3" "plotly>=5,<7" "openai>=1.55,<3"

In [2]:
!pip install -q python-dotenv

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

## Connect (provided helper; only your team tables are writable)

In [5]:
import os, getpass
from contextlib import closing
import psycopg
from psycopg import sql
import pandas as pd

# Reenter these in EACH notebook. Use the team credential supplied privately by your instructor.
PROJECT_REF = os.getenv("LAB_PROJECT_REF") or input("Supabase project ref: ").strip()
POOLER_HOST = os.getenv("LAB_POOLER_HOST") or input("Transaction pooler host (not full URL): ").strip()
TEAM_NUMBER = 1
TEAM_SCHEMA = f"team_{TEAM_NUMBER:02d}"
DB_PASSWORD = "Io1YBgU7Gx8k00PRDnAI74laQAs"
DB_USER = f"lab_team_{TEAM_NUMBER:02d}.{PROJECT_REF}"

def connect():
    # Supabase Free transaction pooler; psycopg prepared statements disabled.
    return psycopg.connect(host=POOLER_HOST, port=6543, dbname="postgres",
                           user=DB_USER, password=DB_PASSWORD, sslmode="require",
                           connect_timeout=12, prepare_threshold=None)

def fetch_df(query, params=None):
    with closing(connect()) as conn:
        with conn.cursor() as cur:
            cur.execute(query, params)
            if cur.description is None: return pd.DataFrame()
            return pd.DataFrame(cur.fetchall(), columns=[x.name for x in cur.description])

def run_sql(query, params=None):
    with closing(connect()) as conn:
        with conn.transaction():
            with conn.cursor() as cur:
                cur.execute(query, params)

print("Connected to:", TEAM_SCHEMA)
print(fetch_df("SELECT current_user AS user_name, current_database() AS db_name"))

Connected to: team_01
     user_name   db_name
0  lab_team_01  postgres


## Explore schema before changing data
**Exercise:** Name one primary key, one foreign key, and one reason the team has its own schema. The instructor has already created the tables—do not run DDL.

In [6]:
display(fetch_df("""SELECT table_schema, table_name, column_name, data_type
  FROM information_schema.columns
  WHERE (table_schema='shared' AND table_name IN ('raw_tickets','ticket_types','knowledge_articles'))
     OR (table_schema=%s AND table_name IN ('clean_tickets','knowledge_articles'))
  ORDER BY table_schema, table_name, ordinal_position""",(TEAM_SCHEMA,)).head(45))

,table_schema,table_name,column_name,data_type
0,shared,knowledge_articles,article_id,text
1,shared,knowledge_articles,title,text
2,shared,knowledge_articles,category,text
3,shared,knowledge_articles,content,text
4,shared,knowledge_articles,embedding,USER-DEFINED
5,shared,raw_tickets,ticket_id,integer
6,shared,raw_tickets,ticket_type,text
7,shared,raw_tickets,ticket_subject,text
8,shared,raw_tickets,ticket_description,text
9,shared,raw_tickets,status,text


In [7]:
raw = fetch_df("SELECT * FROM shared.raw_tickets ORDER BY ticket_id LIMIT 600")
print("Raw records:",len(raw)); display(raw.head(3))
print("Null counts:"); display(raw.isna().sum().to_frame('nulls').T)

Raw records: 600


,ticket_id,ticket_type,ticket_subject,ticket_description,status,priority,channel,product,first_response_at,resolved_at,satisfaction_rating,resolution_text
0,1,Technical issue,Product setup,I'm having an issue with the [product]. Please...,Pending Customer Response,Critical,Social media,GoPro Hero,2023-06-01 12:15:36+00:00,NaT,None,None
1,2,Technical issue,Peripheral compatibility,I'm having an issue with the [product]. Please...,Pending Customer Response,Critical,Chat,LG Smart TV,2023-06-01 16:45:38+00:00,NaT,None,None
2,3,Technical issue,Network problem,I'm facing a problem with my [product]. The [p...,Closed,Low,Social media,Dell XPS,2023-06-01 11:14:38+00:00,2023-06-01 18:05:38+00:00,3.00,Case maybe show recently my computer follow.


Null counts:


,ticket_id,ticket_type,ticket_subject,ticket_description,status,priority,channel,product,first_response_at,resolved_at,satisfaction_rating,resolution_text
nulls,0,0,0,0,0,0,0,0,191,392,392,392


## TODO 1 · Clean the tickets
Implement **three targeted expressions**; leave the rest unchanged. Filter rows with missing ticket identifiers or ticket types, normalize string fields, and deduplicate. Never impute a missing resolution timestamp as zero hours.

In [8]:
df = raw.copy()
df["ticket_id"] = pd.to_numeric(df["ticket_id"], errors="coerce")
df["ticket_type"] = df["ticket_type"].astype("string").str.strip()
df["status"] = df["status"].astype("string").str.strip().str.title()

# TODO 1: Remove rows with missing ticket ID or ticket type
df = df.dropna(subset=["ticket_id", "ticket_type"])
# TODO 2: Keep one row for each ticket ID
df = df.drop_duplicates(subset=["ticket_id"], keep="last")

for col in ("first_response_at", "resolved_at"):
    df[col] = pd.to_datetime(df[col],utc=True,errors="coerce")
df["satisfaction_rating"] = pd.to_numeric(df["satisfaction_rating"],errors="coerce")

# TODO 3: Compute hours from first response to resolution (not total ticket lifetime)
hours = (df["resolved_at"] - df["first_response_at"]).dt.total_seconds() / 3600
df["response_to_resolution_hours"] = hours.where(
    df["status"].eq("Closed") & hours.between(0, 24*365)
).round(2)

df = df.astype(object).where(pd.notna(df), None)
print("Cleaned tickets:",len(df))
display(df[["ticket_id","ticket_type","status","response_to_resolution_hours"]].head())

Cleaned tickets: 600


,ticket_id,ticket_type,status,response_to_resolution_hours
0,1,Technical issue,Pending Customer Response,None
1,2,Technical issue,Pending Customer Response,None
2,3,Technical issue,Closed,6.85
3,4,Billing inquiry,Closed,None
4,5,Billing inquiry,Closed,19.68


## Load into your team table — prewritten UPSERT

In [9]:
# Provided loader. Idempotent UPSERT makes reruns safe.
from datetime import datetime
columns = ["ticket_id","ticket_type","ticket_subject","status","priority","channel", "product",
           "first_response_at","resolved_at","satisfaction_rating","response_to_resolution_hours"]
allowed_types = set(fetch_df("SELECT ticket_type FROM shared.ticket_types")["ticket_type"])
ready = df[columns].copy()
ready = ready[ready["ticket_type"].isin(allowed_types)]
ready["ticket_id"] = ready["ticket_id"].astype(int)

def pgval(v):
    if v is None or (not isinstance(v, (list,tuple,dict)) and pd.isna(v)): return None
    if hasattr(v,"to_pydatetime"): return v.to_pydatetime()
    return v

placeholders=','.join(['%s']*len(columns))
updates=','.join(f"{c}=EXCLUDED.{c}" for c in columns if c != "ticket_id")
query=f"""INSERT INTO {TEAM_SCHEMA}.clean_tickets ({','.join(columns)})
VALUES ({placeholders}) ON CONFLICT(ticket_id) DO UPDATE SET {updates}"""
with closing(connect()) as conn:
    with conn.transaction():
        with conn.cursor() as cur:
            cur.executemany(query,[tuple(pgval(v) for v in row) for row in ready.itertuples(index=False,name=None)])
print("Loaded:",len(ready),"team rows")
assert int(fetch_df(f"SELECT COUNT(*) AS n FROM {TEAM_SCHEMA}.clean_tickets").iloc[0,0]) >= 1

Loaded: 600 team rows


## TODO 4 · Business SQL: JOIN and aggregation

In [10]:
# TODO 4: Complete JOIN and GROUP BY (no instructor DDL needed).
q = f"""
SELECT t.ticket_type, COUNT(*) AS total_tickets,
       ROUND(AVG(t.response_to_resolution_hours), 2) AS avg_response_to_resolution_hours
FROM {TEAM_SCHEMA}.clean_tickets t
JOIN shared.ticket_types c ON t.ticket_type = c.ticket_type
GROUP BY t.ticket_type
ORDER BY total_tickets DESC
"""
display(fetch_df(q))

,ticket_type,total_tickets,avg_response_to_resolution_hours
0,Technical issue,132,7.26
1,Cancellation request,124,7.79
2,Refund request,123,9.14
3,Billing inquiry,112,8.45
4,Product inquiry,109,6.15


## TODO 5 · Closed rate (a business metric, not official SLA)

In [11]:
# TODO 5: Use FILTER to count closed tickets and calculate closed rate.
q = f"""
SELECT ticket_type,
       COUNT(*) AS total,
       COUNT(*) FILTER (WHERE status = 'Closed') AS closed,
       ROUND(100.0 * COUNT(*) FILTER (WHERE status = 'Closed') / NULLIF(COUNT(*),0), 1) AS closed_pct
FROM {TEAM_SCHEMA}.clean_tickets
GROUP BY ticket_type ORDER BY total DESC
"""
display(fetch_df(q))

,ticket_type,total,closed,closed_pct
0,Technical issue,132,49,37.1
1,Cancellation request,124,40,32.3
2,Refund request,123,46,37.4
3,Billing inquiry,112,39,34.8
4,Product inquiry,109,34,31.2


### Checkpoint
You should see results for each ticket type, and the team-owned table should contain cleaned rows. If something fails, ask the instructor to share a solution cell and re-run—the subsequent notebooks read this same table.

In [12]:
# Recovery checkpoint: use this ONLY if your ETL was blocked.
# The instructor can run the same cleaning logic and load a reference snapshot.
RECOVER_FROM_CHECKPOINT = False
if RECOVER_FROM_CHECKPOINT:
    print("Ask the instructor to run the reference solution before continuing.")

print("Verified rows:", int(fetch_df(f"SELECT COUNT(*) AS n FROM {TEAM_SCHEMA}.clean_tickets").iloc[0,0]))

Verified rows: 600
